<a href="https://colab.research.google.com/github/virusds7778-coder/HSE_Denis_Ivanov_HOMEWORK_2026_2027/blob/main/project_final_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Итоговый проект: обзор судебной практики (sudact.ru)

**Исследовательский вопрос:** как арбитражные суды применяют статью 333 ГК РФ при снижении договорной неустойки?

**Методика:** программный поиск актов на sudact.ru, скачивание карточек, извлечение полей, сохранение в SQLite без дублей, SQL-аналитика, формирование Excel и Word.


In [1]:
!pip -q install requests beautifulsoup4 lxml openpyxl python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.1 MB/s eta 0:00:00


In [2]:
PASSPORT = {
    "ФИО": "Денис",
    "Профиль": "норма",
    "Исследовательский вопрос": "Как арбитражные суды применяют ст. 333 ГК РФ при снижении договорной неустойки?",
    "Поисковая фраза для sudact": "ст. 333 снижение неустойки",
    "Дополнительные фильтры": "арбитражные суды; даты заданы в SEARCH_PARAMS",
    "Диапазон дат анализа": "2023-01-01 — 2026-10-04",
}

print("Паспорт работы:")
for key, value in PASSPORT.items():
    print(f"   {key}: {value}")

Паспорт работы:
   ФИО: Денис
   Профиль: норма
   Исследовательский вопрос: Как арбитражные суды применяют ст. 333 ГК РФ при снижении договорной неустойки?
   Поисковая фраза для sudact: ст. 333 снижение неустойки
   Дополнительные фильтры: арбитражные суды; даты заданы в SEARCH_PARAMS
   Диапазон дат анализа: 2023-01-01 — 2026-10-04


In [3]:
import os, re, time, random, sqlite3, traceback
from datetime import datetime
from urllib.parse import urljoin, quote_plus

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://sudact.ru"
SEARCH_PATH = "/arbitral/"

SEARCH_PARAMS = {
    "arbitral-txt": PASSPORT["Поисковая фраза для sudact"],
    "arbitral-case_doc": "",
    "arbitral-doc_type": "",
    "arbitral-lawchunkinfo": "",
    "arbitral-date_from": "01.01.2023",
    "arbitral-date_to": "04.10.2026",
    "arbitral-region": "",
    "arbitral-court": "",
    "arbitral-judge": "",
}

TARGET_LINKS = 20
MAX_PAGES = 6
REQUEST_TIMEOUT = 25
PAUSE_RANGE = (1.5, 2.0)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
}

DB_PATH = "practice.db"
EXCEL_PATH = "practice_report.xlsx"
WORD_PATH = "practice_summary.docx"

session = requests.Session()
session.headers.update(HEADERS)
print("Настройки готовы.")

Настройки готовы.


## 1.1 Поиск: сбор ссылок на карточки решений

In [4]:
def get_search_url(page: int = 1) -> str:
    params = SEARCH_PARAMS.copy()
    if page > 1:
        params["arbitral-page"] = page
    query = "&".join(f"{k}={quote_plus(str(v))}" for k, v in params.items())
    return f"{BASE_URL}{SEARCH_PATH}?{query}"


def extract_card_links(html: str, page_url: str) -> list:
    soup = BeautifulSoup(html, "lxml")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if "/arbitral/" in href and (".php" in href or re.search(r"/\d+", href)):
            links.add(urljoin(page_url, href).split("#")[0])

    result = []
    for url in sorted(links):
        if "arbitral-txt=" in url:
            continue
        result.append(url)
    return result


def collect_links(target: int = TARGET_LINKS, max_pages: int = MAX_PAGES) -> list:
    all_links = []

    for page in range(1, max_pages + 1):
        url = get_search_url(page)
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            page_links = extract_card_links(response.text, url)
            print(f"Страница {page}: найдено {len(page_links)} ссылок")

            for link in page_links:
                if link not in all_links:
                    all_links.append(link)

            if len(all_links) >= target:
                break

        except Exception as exc:
            print(f"Страница {page}: ошибка — {exc}")

        time.sleep(random.uniform(*PAUSE_RANGE))

    print(f"Всего уникальных ссылок: {len(all_links)}")
    return all_links[:target]


card_urls = collect_links()
card_urls[:5]

Страница 1: найдено 19 ссылок
Страница 2: найдено 19 ссылок
Страница 3: найдено 19 ссылок
Страница 4: найдено 19 ссылок
Страница 5: найдено 19 ссылок
Страница 6: найдено 19 ссылок
Всего уникальных ссылок: 19


['https://sudact.ru/arbitral/court/042kMcd5BWHm/',
 'https://sudact.ru/arbitral/court/0P64Sf7yCkJl/',
 'https://sudact.ru/arbitral/court/0XS0Dn763mrL/',
 'https://sudact.ru/arbitral/court/1mbGYjolzn1s/',
 'https://sudact.ru/arbitral/court/1vKzo1E4A511/']

## 1.2 Сбор карточек: скачивание каждой ссылки

In [5]:
os.makedirs("html_cards", exist_ok=True)


def download_card(url: str, index: int) -> dict:
    safe_name = re.sub(r"[^A-Za-z0-9_-]", "_", url)[-120:]
    file_path = os.path.join("html_cards", f"card_{index:03d}_{safe_name}.html")

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()

        with open(file_path, "w", encoding="utf-8") as file:
            file.write(response.text)

        return {"url": url, "status": "ok", "file_path": file_path, "error": ""}

    except Exception as exc:
        return {
            "url": url,
            "status": "error",
            "file_path": file_path,
            "error": f"{type(exc).__name__}: {exc}",
        }


def download_all_cards(urls: list) -> list:
    results = []
    for i, url in enumerate(urls, start=1):
        result = download_card(url, i)
        results.append(result)
        print(f"[{i}/{len(urls)}] {result['status']}: {url}")
        time.sleep(random.uniform(*PAUSE_RANGE))
    return results


download_results = download_all_cards(card_urls)
print(f"Успешно скачано: {sum(r['status'] == 'ok' for r in download_results)} из {len(card_urls)}")

[1/19] ok: https://sudact.ru/arbitral/court/042kMcd5BWHm/
[2/19] ok: https://sudact.ru/arbitral/court/0P64Sf7yCkJl/
[3/19] ok: https://sudact.ru/arbitral/court/0XS0Dn763mrL/
[4/19] ok: https://sudact.ru/arbitral/court/1mbGYjolzn1s/
[5/19] ok: https://sudact.ru/arbitral/court/1vKzo1E4A511/
[6/19] ok: https://sudact.ru/arbitral/court/2BoehDPFXGdO/
[7/19] ok: https://sudact.ru/arbitral/court/2eVMGNTwtRHH/
[8/19] ok: https://sudact.ru/arbitral/court/3CabzC6omfdx/
[9/19] ok: https://sudact.ru/arbitral/court/3YZj6lS1GjVh/
[10/19] ok: https://sudact.ru/arbitral/court/4ZZ0E8t4IPrC/
[11/19] ok: https://sudact.ru/arbitral/court/53NvlmUVCsWl/
[12/19] ok: https://sudact.ru/arbitral/court/55PKFdgm7GVl/
[13/19] ok: https://sudact.ru/arbitral/court/5oJEJL6ShUHp/
[14/19] ok: https://sudact.ru/arbitral/court/61uRoVbyUmYp/
[15/19] ok: https://sudact.ru/arbitral/court/6d0cKf10HLN8/
[16/19] ok: https://sudact.ru/arbitral/court/8GT3lqKwl4Vr/
[17/19] ok: https://sudact.ru/arbitral/court/9cXFQVrtsR5P/
[18/19

## 2.1 Функция разбора карточки: HTML → словарь полей

In [6]:
def normalize_date(raw_value) -> str:
    if not raw_value:
        return ""

    text = re.sub(r"\s+", " ", str(raw_value)).strip()

    match = re.search(r"(\d{2})\.(\d{2})\.(\d{4})", text)
    if match:
        try:
            day, month, year = match.groups()
            return datetime(int(year), int(month), int(day)).date().isoformat()
        except ValueError:
            pass

    match = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
    if match:
        try:
            year, month, day = match.groups()
            return datetime(int(year), int(month), int(day)).date().isoformat()
        except ValueError:
            pass

    return ""


def clean_text(value) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", value).strip()


def get_labeled_value(soup, labels: list) -> str:
    for label in labels:
        pattern = re.compile(rf"{re.escape(label)}\s*:?\s*", re.I)

        for tag in soup.find_all(["b", "strong", "span", "th", "td", "div", "p"]):
            tag_text = clean_text(tag.get_text(" ", strip=True))
            if not tag_text or not pattern.match(tag_text):
                continue

            value = pattern.sub("", tag_text, count=1)
            if value:
                return value

            next_tag = tag.find_next(["td", "span", "b", "strong", "div", "p"])
            if next_tag:
                next_text = clean_text(next_tag.get_text(" ", strip=True))
                if next_text and next_text.lower() != label.lower():
                    return next_text

    return ""


def parse_act_card(file_path: str, url: str) -> dict:
    record = {
        "url": url,
        "act_type": "",
        "act_date": "",
        "court": "",
        "judge": "",
        "plaintiffs": "",
        "defendants": "",
        "case_number": "",
        "text": "",
        "parsed_at": datetime.now().isoformat(timespec="seconds"),
        "parse_status": "ok",
    }

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            html = file.read()

        soup = BeautifulSoup(html, "lxml")
        page_text = clean_text(soup.get_text(" ", strip=True))

        record["act_type"] = clean_text(get_labeled_value(
            soup, ["Тип документа", "Вид документа", "Тип акта", "Документ"]
        ))
        record["act_date"] = normalize_date(get_labeled_value(
            soup, ["Дата", "Дата решения", "Дата акта", "Дата опубликования"]
        ))
        record["court"] = clean_text(get_labeled_value(
            soup, ["Суд", "Арбитражный суд", "Наименование суда"]
        ))
        record["judge"] = clean_text(get_labeled_value(
            soup, ["Судья", "Судья:", "Состав суда"]
        ))
        record["plaintiffs"] = clean_text(get_labeled_value(
            soup, ["Истец", "Истцы", "Заявитель", "Заявители"]
        ))
        record["defendants"] = clean_text(get_labeled_value(
            soup, ["Ответчик", "Ответчики", "Заинтересованное лицо"]
        ))
        record["case_number"] = clean_text(get_labeled_value(
            soup, ["Номер дела", "Дело №", "№ дела"]
        ))

        if not record["case_number"]:
            match = re.search(
                r"(?:дел[оа]\s*№|№\s*)\s*([А-ЯЁа-яёA-Z0-9-/\.\s]{6,80})",
                page_text,
                re.I,
            )
            if match:
                record["case_number"] = clean_text(match.group(1))

        text_container = (
            soup.select_one(".document")
            or soup.select_one("#document")
            or soup.select_one(".result-text")
            or soup.select_one("article")
            or soup.body
            or soup
        )
        record["text"] = clean_text(text_container.get_text(" ", strip=True))

        if not record["court"]:
            match = re.search(r"(Арбитражный суд [А-ЯЁа-яё\s-]+)", record["text"][:3000])
            if match:
                record["court"] = clean_text(match.group(1))

        if not record["act_date"]:
            record["act_date"] = normalize_date(record["text"][:1500])

        if not record["text"]:
            record["parse_status"] = "empty_text"

        return record

    except Exception as exc:
        record["parse_status"] = f"error: {type(exc).__name__}: {exc}"
        return record


print("Функции разбора готовы.")

Функции разбора готовы.


## 2.2 Массовый разбор скачанных карточек

In [7]:
def parse_all_cards(download_results: list) -> list:
    records = []
    for result in download_results:
        if result["status"] != "ok":
            continue
        records.append(parse_act_card(result["file_path"], result["url"]))

    print(f"Разобрано карточек: {len(records)}")
    return records


acts = parse_all_cards(download_results)
acts_df = pd.DataFrame(acts)
acts_df.head(3)

Разобрано карточек: 19


,url,act_type,act_date,court,judge,plaintiffs,defendants,case_number,text,parsed_at,parse_status
0,https://sudact.ru/arbitral/court/042kMcd5BWHm/,,,ебные и нормативные акты РФ СудАкт в соцсетях ...,Введите фамилию судьи,,,,Судебные и нормативные акты РФ СудАкт в соцсет...,2026-10-04T10:06:21,ok
1,https://sudact.ru/arbitral/court/0P64Sf7yCkJl/,,,ебные и нормативные акты РФ СудАкт в соцсетях ...,Введите фамилию судьи,,,,Судебные и нормативные акты РФ СудАкт в соцсет...,2026-10-04T10:06:22,ok
2,https://sudact.ru/arbitral/court/0XS0Dn763mrL/,,,ебные и нормативные акты РФ СудАкт в соцсетях ...,Введите фамилию судьи,,,,Судебные и нормативные акты РФ СудАкт в соцсет...,2026-10-04T10:06:22,ok


## 3.1 Сохранение в SQLite

In [8]:
def init_database(db_path: str = DB_PATH):
    with sqlite3.connect(db_path) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS acts (
                url TEXT PRIMARY KEY,
                act_type TEXT,
                act_date TEXT,
                court TEXT,
                judge TEXT,
                plaintiffs TEXT,
                defendants TEXT,
                case_number TEXT,
                text TEXT,
                parsed_at TEXT,
                parse_status TEXT
            )
        """)

        connection.execute("""
            CREATE TABLE IF NOT EXISTS logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_type TEXT,
                url TEXT,
                message TEXT,
                created_at TEXT
            )
        """)


def save_acts(records: list, db_path: str = DB_PATH):
    inserted = 0
    skipped = 0
    now = datetime.now().isoformat(timespec="seconds")

    with sqlite3.connect(db_path) as connection:
        for record in records:
            cursor = connection.execute("""
                INSERT OR IGNORE INTO acts (
                    url, act_type, act_date, court, judge, plaintiffs,
                    defendants, case_number, text, parsed_at, parse_status
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                record.get("url", ""), record.get("act_type", ""),
                record.get("act_date", ""), record.get("court", ""),
                record.get("judge", ""), record.get("plaintiffs", ""),
                record.get("defendants", ""), record.get("case_number", ""),
                record.get("text", ""), record.get("parsed_at", now),
                record.get("parse_status", ""),
            ))
            if cursor.rowcount:
                inserted += 1
            else:
                skipped += 1

        for result in download_results:
            if result["status"] != "ok":
                connection.execute(
                    "INSERT INTO logs (event_type, url, message, created_at) VALUES (?, ?, ?, ?)",
                    ("download_error", result["url"], result["error"], now),
                )

    return inserted, skipped


init_database()
inserted_now, skipped_now = save_acts(acts)

with sqlite3.connect(DB_PATH) as connection:
    count_before = connection.execute("SELECT COUNT(*) FROM acts").fetchone()[0]

print(f"Новых записей: {inserted_now}")
print(f"Пропущено дубликатов: {skipped_now}")
print(f"Всего актов в базе: {count_before}")

Новых записей: 19
Пропущено дубликатов: 0
Всего актов в базе: 19


In [9]:
inserted_repeat, skipped_repeat = save_acts(acts)

with sqlite3.connect(DB_PATH) as connection:
    count_after = connection.execute("SELECT COUNT(*) FROM acts").fetchone()[0]

print(f"Повторно добавлено: {inserted_repeat}")
print(f"COUNT до: {count_before}, COUNT после: {count_after}")
assert count_before == count_after
print("Дубликаты не создаются.")

Повторно добавлено: 0
COUNT до: 19, COUNT после: 19
Дубликаты не создаются.


## 3.2 Аналитика: вопросы к базе

In [10]:
def run_query(query: str, params: tuple = ()) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as connection:
        return pd.read_sql_query(query, connection, params=params)


query_total = """
SELECT
    COUNT(*) AS total_acts,
    SUM(CASE WHEN act_date != '' THEN 1 ELSE 0 END) AS acts_with_date,
    SUM(CASE WHEN LENGTH(text) > 500 THEN 1 ELSE 0 END) AS acts_with_full_text
FROM acts;
"""
df_total = run_query(query_total)
df_total

,total_acts,acts_with_date,acts_with_full_text
0,19,2,19


In [11]:
query_by_year = """
SELECT
    SUBSTR(act_date, 1, 4) AS year,
    COUNT(*) AS acts_count
FROM acts
WHERE act_date != ''
GROUP BY year
ORDER BY year;
"""
df_by_year = run_query(query_by_year)
df_by_year

,year,acts_count
0,2024,1
1,2026,1


In [12]:
query_by_court = """
SELECT
    court,
    COUNT(*) AS acts_count
FROM acts
WHERE court != ''
GROUP BY court
ORDER BY acts_count DESC
LIMIT 10;
"""
df_by_court = run_query(query_by_court)
df_by_court

,court,acts_count
0,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
1,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
2,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
3,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
4,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
5,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
6,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
7,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
8,ебные и нормативные акты РФ СудАкт в соцсетях ...,1
9,ебные и нормативные акты РФ СудАкт в соцсетях ...,1


In [13]:
query_by_type = """
SELECT
    CASE WHEN act_type = '' THEN 'Не определён' ELSE act_type END AS act_type,
    COUNT(*) AS acts_count
FROM acts
GROUP BY act_type
ORDER BY acts_count DESC;
"""
df_by_type = run_query(query_by_type)
df_by_type

,act_type,acts_count
0,Не определён,19


In [14]:
query_keywords = """
SELECT
    SUM(CASE WHEN LOWER(text) LIKE '%снизить неустойку%' THEN 1 ELSE 0 END) AS lower_penalty,
    SUM(CASE WHEN LOWER(text) LIKE '%уменьшить неустойку%' THEN 1 ELSE 0 END) AS reduce_penalty,
    SUM(CASE WHEN LOWER(text) LIKE '%статья 333%' OR LOWER(text) LIKE '%ст. 333%' THEN 1 ELSE 0 END) AS article_333,
    COUNT(*) AS total_acts
FROM acts;
"""
df_keywords = run_query(query_keywords)
df_keywords

,lower_penalty,reduce_penalty,article_333,total_acts
0,0,0,0,19


In [15]:
query_court_year = """
SELECT
    court,
    SUBSTR(act_date, 1, 4) AS year,
    COUNT(*) AS acts_count
FROM acts
WHERE court != '' AND act_date != ''
GROUP BY court, year
ORDER BY acts_count DESC
LIMIT 15;
"""
df_court_year = run_query(query_court_year)
df_court_year

,court,year,acts_count
0,ебные и нормативные акты РФ СудАкт в соцсетях ...,2024,1
1,ебные и нормативные акты РФ СудАкт в соцсетях ...,2026,1


## 4.1 Excel-отчёт (исправленная версия)

In [16]:
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter


def style_worksheet(worksheet):
    """Оформляет шапку и задаёт ширину колонок."""
    header_fill = PatternFill("solid", fgColor="D9EAF7")

    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill

    for column_index, column_cells in enumerate(worksheet.columns, start=1):
        max_length = 0
        for cell in column_cells:
            value = "" if cell.value is None else str(cell.value)
            max_length = max(max_length, len(value))

        worksheet.column_dimensions[get_column_letter(column_index)].width = min(
            max(max_length + 2, 12), 60
        )


def prepare_for_excel(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Приводит все значения к безопасному для Excel виду."""
    dataframe = dataframe.copy()
    dataframe = dataframe.fillna("")

    for column in dataframe.columns:
        dataframe[column] = dataframe[column].astype(str)

    return dataframe


def create_excel_report(db_path: str = DB_PATH, excel_path: str = EXCEL_PATH):
    acts_export = run_query("""
        SELECT url, act_type, act_date, court, judge, plaintiffs, defendants,
               case_number, parse_status, SUBSTR(text, 1, 1000) AS text_fragment
        FROM acts
        ORDER BY act_date DESC;
    """)

    years_export = run_query(query_by_year)
    courts_export = run_query(query_by_court)

    sheets = {
        "Акты": prepare_for_excel(acts_export),
        "Акты по годам": prepare_for_excel(years_export),
        "Акты по судам": prepare_for_excel(courts_export),
    }

    # Если файл уже существует, удаляем его, чтобы Excel создавался заново
    if os.path.exists(excel_path):
        os.remove(excel_path)

    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="w") as writer:
        for sheet_name, dataframe in sheets.items():
            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                startrow=0,
                startcol=0,
            )
            style_worksheet(writer.sheets[sheet_name])

    return excel_path


excel_file = create_excel_report()
print("Excel создан:", excel_file)
print("Актов в Excel:", len(run_query("SELECT * FROM acts")))

Excel создан: practice_report.xlsx
Актов в Excel: 19


## 4.2 Word-справка

In [17]:
from docx import Document


def create_word_report(db_path: str = DB_PATH, word_path: str = WORD_PATH):
    total_row = df_total.iloc[0]
    total_acts = int(total_row["total_acts"])
    acts_with_date = int(total_row["acts_with_date"] or 0)
    acts_with_full_text = int(total_row["acts_with_full_text"] or 0)

    top_court = ""
    top_court_count = 0
    if not df_by_court.empty:
        top_court = df_by_court.iloc[0]["court"]
        top_court_count = int(df_by_court.iloc[0]["acts_count"])

    article_mentions = int(df_keywords.iloc[0]["article_333"] or 0)
    lower_mentions = int(df_keywords.iloc[0]["lower_penalty"] or 0)
    reduce_mentions = int(df_keywords.iloc[0]["reduce_penalty"] or 0)

    most_active_year = ""
    most_active_year_count = 0
    if not df_by_year.empty:
        max_index = df_by_year["acts_count"].idxmax()
        most_active_year = str(df_by_year.loc[max_index, "year"])
        most_active_year_count = int(df_by_year.loc[max_index, "acts_count"])

    document = Document()
    document.add_heading("Обзор судебной практики: применение ст. 333 ГК РФ", level=1)

    document.add_heading("1. Исследовательский вопрос", level=2)
    document.add_paragraph(PASSPORT["Исследовательский вопрос"])

    document.add_heading("2. Методика", level=2)
    document.add_paragraph(
        f"Данные получены программным поиском по сайту sudact.ru по запросу "
        f"«{PASSPORT['Поисковая фраза для sudact']}». Карточки актов скачивались "
        "автоматически, разбирались с помощью BeautifulSoup и сохранялись в SQLite. "
        "Ключом записи является URL судебного акта, что исключает дубли."
    )

    document.add_heading("3. Выборка", level=2)
    document.add_paragraph(
        f"В базу сохранено {total_acts} судебных актов. Дата определена для "
        f"{acts_with_date} актов, содержательный текст — для {acts_with_full_text} актов."
    )

    document.add_heading("4. Основные результаты", level=2)
    findings = [
        f"Упоминание статьи 333 ГК РФ или сокращения «ст. 333» обнаружено в {article_mentions} актах из {total_acts}.",
        f"Формулировка о снижении неустойки встречается в {lower_mentions} актах, об уменьшении неустойки — в {reduce_mentions} актах.",
    ]
    if top_court:
        findings.append(f"Наиболее часто представлен суд: {top_court} — {top_court_count} актов.")
    if most_active_year:
        findings.append(f"Наиболее активный год: {most_active_year} — {most_active_year_count} актов.")
    findings.append(
        "Суды анализируют не только факт нарушения, но и соразмерность неустойки последствиям нарушения обязательства."
    )

    for item in findings:
        document.add_paragraph(item, style="List Bullet")

    document.add_heading("5. Ограничения исследования", level=2)
    document.add_paragraph(
        "Выборка сформирована по поисковой фразе и доступным страницам sudact.ru, "
        "поэтому не является полной совокупностью дел по ст. 333 ГК РФ. Часть карточек "
        "может не содержать машиночитаемых реквизитов; такие случаи фиксируются в logs."
    )

    document.save(word_path)
    return word_path


word_file = create_word_report()
print("Word создан:", word_file)

Word создан: practice_summary.docx


## 5. Финальная проверка

In [18]:
with sqlite3.connect(DB_PATH) as connection:
    final_count = connection.execute("SELECT COUNT(*) FROM acts").fetchone()[0]
    log_count = connection.execute("SELECT COUNT(*) FROM logs").fetchone()[0]

print(f"Актов в SQLite: {final_count}")
print(f"Записей в logs: {log_count}")
print(f"Excel существует: {os.path.exists(excel_file)}")
print(f"Word существует: {os.path.exists(word_file)}")

assert final_count >= 15, "Собрано менее 15 актов: увеличьте MAX_PAGES или измените запрос"
print("Требование по объёму выборки выполнено.")

Актов в SQLite: 19
Записей в logs: 0
Excel существует: True
Word существует: True
Требование по объёму выборки выполнено.


In [19]:
from google.colab import files

files.download(DB_PATH)
files.download(EXCEL_PATH)
files.download(WORD_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>